<a href="https://colab.research.google.com/github/kainatff/BioML-Diabetes/blob/main/code/mlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install scikit-learn pandas matplotlib seaborn tensorflow --quiet
!pip install scikeras --quiet

import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, cohen_kappa_score, mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

class MLPDiabetesClassifier:
    def __init__(self):
        self.scaler = StandardScaler()

    def load_data(self, dataset_url):
        """Load and preprocess diabetes dataset directly from GitHub"""
        try:
            # Convert regular GitHub URL to raw content URL
            if 'github.com' in dataset_url and 'raw.githubusercontent.com' not in dataset_url:
                dataset_url = dataset_url.replace('github.com', 'raw.githubusercontent.com').replace('blob/', '')

            data = pd.read_csv(dataset_url)

            # Rename outcome column if needed
            if 'Outcome' in data.columns:
                data.rename(columns={'Outcome': 'class'}, inplace=True)

            X = data.iloc[:, :-1].values
            y = data.iloc[:, -1].values

            X = self.scaler.fit_transform(X)
            return X, y
        except Exception as e:
            print(f"Error loading data from GitHub: {e}")
            return None, None

    def build_model(self, input_dim):
        """Build and compile the MLP model"""
        model = Sequential([
             Dense(32, activation='relu', input_shape=(input_dim,)),
             Dense(16, activation='relu'),
             Dense(1, activation='sigmoid')  # Binary output
          ])
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

        return model

    def evaluate_mlp(self, X_train, X_test, y_train, y_test, selected_features):
        """Train and evaluate MLP model on selected features"""
        feature_idx = [f - 1 for f in selected_features]
        X_train_sel = X_train[:, feature_idx]
        X_test_sel = X_test[:, feature_idx]

        start_time = time.time()

        # Build and train the main model
        model = self.build_model(input_dim=len(feature_idx))
        history = model.fit(
            X_train_sel, y_train,
            epochs=100,
            batch_size=16,
            verbose=0,
            validation_split=0.2
        )

        # Make predictions
        y_pred_prob = model.predict(X_test_sel).flatten()
        y_pred = (y_pred_prob > 0.5).astype(int)

        # Calculate metrics
        acc = accuracy_score(y_test, y_pred)
        kappa = cohen_kappa_score(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)

        elapsed = time.time() - start_time

        return {
            'accuracy': acc,
            'kappa': kappa,
            'mae': mae,
            'time': elapsed,
            'history': history.history
        }

    def run_mlp_experiment(self, dataset_url, dataset_name, selected_features):
        """Run the MLP experiment on a single dataset"""
        print(f"\n{'='*50}")
        print(f"Running MLP on {dataset_name} Dataset")
        print(f"Selected Features: {selected_features}")
        print(f"{'='*50}")

        # Load data
        X, y = self.load_data(dataset_url)
        if X is None:
            return None

        # Split data (70% train, 30% test)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.3, random_state=42
        )

        # Evaluate MLP
        results = self.evaluate_mlp(X_train, X_test, y_train, y_test, selected_features)

        # Print results
        print("\nMLP Performance:")
        print("{:<10} {:<10} {:<10} {:<10} {:<10}".format(
            'Metric', 'Accuracy', 'Kappa', 'MAE', 'Time(s)'
        ))
        print("-" * 60)
        print("{:<10} {:<10.4f} {:<10.4f} {:<10.4f} {:<10.4f}".format(
            'MLP',
            results['accuracy'],
            results['kappa'],
            results['mae'],
            results['time']
        ))

        return results

if __name__ == "__main__":
    # Initialize classifier
    mlp_classifier = MLPDiabetesClassifier()

    # GitHub dataset URLs and selected features
    experiments = [
        {
            'name': 'PID',
            'url': 'https://github.com/kainatff/BioML-Diabetes/blob/main/diabetes.csv',
            'features': [1, 2, 6, 7]  # Selected features for PID dataset
        },
        {
            'name': 'HFD',
            'url': 'https://github.com/kainatff/BioML-Diabetes/blob/main/diabetes2.csv',
            'features': [2, 3, 6, 8]  # Selected features for HFD dataset
        }
    ]

    all_results = {}

    # Run experiments
    for exp in experiments:
        results = mlp_classifier.run_mlp_experiment(
            exp['url'], exp['name'], exp['features']
        )
        if results is not None:
            all_results[exp['name']] = results

    # Print summary comparison
    if all_results:
        print("\n\nMLP Performance Summary Across Datasets:")
        print("="*75)
        print("{:<10} {:<15} {:<15} {:<15} {:<15}".format(
            'Dataset', 'Accuracy', 'Kappa', 'MAE', 'Time(s)'
        ))
        print("-"*75)

        for dataset_name, metrics in all_results.items():
            print("{:<10} {:<15.4f} {:<15.4f} {:<15.4f} {:<15.4f}".format(
                dataset_name,
                metrics['accuracy'],
                metrics['kappa'],
                metrics['mae'],
                metrics['time']
            ))


Running MLP on PID Dataset
Selected Features: [1, 2, 6, 7]


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 

MLP Performance:
Metric     Accuracy   Kappa      MAE        Time(s)   
------------------------------------------------------------
MLP        0.7403     0.4160     0.2597     20.8919   

Running MLP on HFD Dataset
Selected Features: [2, 3, 6, 8]


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

MLP Performance:
Metric     Accuracy   Kappa      MAE        Time(s)   
------------------------------------------------------------
MLP        0.8167     0.5926     0.1833     33.4594   


MLP Performance Summary Across Datasets:
Dataset    Accuracy        Kappa           MAE             Time(s)        
---------------------------------------------------------------------------
PID        0.7403          0.4160          0.2597          20.8919        
HFD        0.8167          0.5926          0.1833          33.4594        
